<a href="https://colab.research.google.com/github/sadwi18/24P31A4236_DriveReady_OPC_2026/blob/main/Day_2/day2_shopsphere.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai

import json
import os
from functools import wraps
from google.colab import userdata
from openai import OpenAI, RateLimitError, APIConnectionError

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

client = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

MODEL = "openai/gpt-oss-20b"

CATALOG = {
    "wireless earbuds": 1499,
    "smartwatch": 2499,
    "laptop backpack": 999,
}

STRUCTURED_SYSTEM_PROMPT = """
ROLE:
You are the customer support assistant for ShopSphere Online Store.

FACTS:
- Catalog prices: Wireless Earbuds (Rs. 1499), Smartwatch (Rs. 2499), Laptop Backpack (Rs. 999).
- Returns: 7-day return/replacement window for unused items in original packaging.
- Shipping: Standard delivery in 3-5 business days; free shipping on orders Rs. 1500 and above.
- Payments: UPI, cards, and Cash on Delivery (COD).

RULES:
- Only state prices, policies, and shipping details from FACTS.
- Do not guess order status or unconfirmed delivery guarantees; say you will check with the support team.
- Keep replies concise and polite.

FORMAT:
- Plain text response, under 60 words.
""".strip()

In [5]:
few_shot_messages = [
    {"role": "system", "content": "Classify the customer message into one label only: order_inquiry, return_request, or product_question."},
    {"role": "user", "content": "Where is my parcel #SH2091?"},
    {"role": "assistant", "content": "order_inquiry"},
    {"role": "user", "content": "The smartwatch strap doesn't fit, can I send it back?"},
    {"role": "assistant", "content": "return_request"},
    {"role": "user", "content": "Does the laptop backpack fit a 15-inch laptop?"},
    {"role": "assistant", "content": "product_question"},
    {"role": "user", "content": "I want to return my wireless earbuds bought 3 days ago."},
]

res_cls = client.chat.completions.create(
    model=MODEL,
    messages=few_shot_messages,
    temperature=0.0,
    max_tokens=200,
)
print("Predicted label:", res_cls.choices[0].message.content.strip())

Finish reason: stop
Predicted label: return_request


In [7]:
import re

res_json = client.chat.completions.create(
    model=MODEL,
    messages=extract_prompt,
    temperature=0.0,
    max_tokens=300,
)
raw_text = (res_json.choices[0].message.content or "").strip()

# Strip ```json ... ``` fences if present
cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_text, flags=re.DOTALL).strip()

# Fallback: grab substring from first '{' to last '}' if extra text appeared
match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
json_str = match.group(0) if match else cleaned

parsed_order = json.loads(json_str)
print("Parsed dict:", parsed_order)

Parsed dict: {'product': 'wireless earbuds', 'qty': 2, 'note': 'gift wrap if possible.'}


In [8]:
from functools import wraps


def log_call(func):
    """Log function name and arguments before execution."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] Calling {func.__name__} with args={args}, kwargs={kwargs}")
        return func(*args, **kwargs)
    return wrapper


@log_call
def get_item_price(item: str, qty: int = 1) -> int | str:
    """Return total price in INR for a known ShopSphere item and quantity."""
    key = item.strip().lower()
    if key not in CATALOG:
        return f"Item '{item}' not found in catalog."
    if qty < 1:
        return "Quantity must be at least 1."
    return CATALOG[key] * qty


print("Total for 2 backpacks:", get_item_price("laptop backpack", 2))
print("Total for 1 smartwatch:", get_item_price("Smartwatch"))

[LOG] Calling get_item_price with args=('laptop backpack', 2), kwargs={}
Total for 2 backpacks: 1998
[LOG] Calling get_item_price with args=('Smartwatch',), kwargs={}
Total for 1 smartwatch: 2499


In [9]:
from openai import APIConnectionError, RateLimitError

MAX_TURNS = 4
history = [{"role": "system", "content": STRUCTURED_SYSTEM_PROMPT}]


def send_chat_turn(user_message: str) -> str:
    """Append user turn, trim history, call Groq safely, and record assistant reply."""
    global history
    history.append({"role": "user", "content": user_message})
    history = [history[0]] + history[1:][-(MAX_TURNS * 2):]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=history,
            temperature=0.3,
            max_tokens=300,
        )
        reply = (response.choices[0].message.content or "").strip()
        history.append({"role": "assistant", "content": reply})
        return reply
    except RateLimitError as e:
        return f"Rate limit reached. Please retry shortly. ({e})"
    except APIConnectionError as e:
        return f"Network connection error while contacting Groq API. ({e})"
    except Exception as e:
        return f"Unexpected error: {e}"


print(send_chat_turn("How much does a smartwatch cost?"))
print(send_chat_turn("Is shipping free on it?"))

The Smartwatch is priced at Rs. 2499.
Yes—since the smartwatch costs Rs. 2499 (above the Rs. 1500 threshold), you’ll get free standard shipping.
